In [97]:
import numpy as np
import matplotlib.pyplot as plt
import os
import random
import pandas as pd

from astropy.io import fits
from astropy.table import Table
from scipy.signal import detrend
from astropy.timeseries import LombScargle

In [98]:
#Data Upload

VVV_data = fits.open('VVV_Sample.fits')

Id_b278, Id_b279 = VVV_data[1].data,VVV_data[11].data

Ks_b278, Ks_b279 = VVV_data[2].data, VVV_data[12].data
Ks_err_b278, Ks_err_b279 = VVV_data[5].data, VVV_data[15].data
Ks_mjd_b278, Ks_mjd_b279 = VVV_data[8].data, VVV_data[18].data

#Removing np.nan 

# Remove missing values (NaNs) from all Ks-related data arrays

def remove_nans(array2d):
    return [row[~np.isnan(row)] for row in array2d]

def _safe_norm(x):
    xmin, xmax = np.min(x), np.max(x)
    return (x - xmin) / (xmax - xmin)

Ks_b278 = remove_nans(Ks_b278)
Ks_b279 = remove_nans(Ks_b279)

Ks_err_b278 = remove_nans(Ks_err_b278)
Ks_err_b279 = remove_nans(Ks_err_b279)

Ks_mjd_b278 = remove_nans(Ks_mjd_b278)
Ks_mjd_b279 = remove_nans(Ks_mjd_b279)

In [99]:
VVV_data.info()

Filename: VVV_Sample.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU      11   ()      
  1  ID_B278       1 ImageHDU         7   (304,)   int64   
  2  KS_B278       1 ImageHDU         8   (523, 304)   float64   
  3  H_B278        1 ImageHDU         8   (32, 304)   float64   
  4  J_B278        1 ImageHDU         8   (25, 304)   float64   
  5  KS_ERR_B278    1 ImageHDU         8   (523, 304)   float64   
  6  H_ERR_B278    1 ImageHDU         8   (32, 304)   float64   
  7  J_ERR_B278    1 ImageHDU         8   (25, 304)   float64   
  8  KS_MJD_B278    1 ImageHDU         8   (523, 304)   float64   
  9  H_MJD_B278    1 ImageHDU         8   (32, 304)   float64   
 10  J_MJD_B278    1 ImageHDU         8   (25, 304)   float64   
 11  ID_B279       1 ImageHDU         7   (286,)   int64   
 12  KS_B279       1 ImageHDU         8   (960, 286)   float64   
 13  H_B279        1 ImageHDU         8   (14, 286)   float64   
 14  J_B279        1

In [100]:
def pdm_theta_fast(t, y, period, nbins=12):

    phase = ((t - t[0]) / period) % 1
    sort_idx = np.argsort(phase)

    phase = phase[sort_idx]
    y = y[sort_idx]

    bins = np.linspace(0, 1, nbins + 1)
    indices = np.searchsorted(phase, bins)

    var_total = np.var(y)
    if var_total == 0:
        return np.inf

    num, den = 0, 0

    for i in range(nbins):
        start, end = indices[i], indices[i+1]
        n = end - start

        if n > 2:
            var_bin = np.var(y[start:end])
            num += (n - 1) * var_bin
            den += (n - 1)

    if den == 0:
        return np.inf

    return num / (den * var_total)


def find_period_pdm_refined(t, y, f0, width=0.1, n_freq=400):

    min_f_local = max(min_f, f0 * (1 - width))
    max_f_local = min(max_f, f0 * (1 + width))

    freqs = np.linspace(min_f_local, max_f_local, n_freq)
    periods = 1 / freqs

    theta = np.array([pdm_theta_fast(t, y, p) for p in periods])

    best_idx = np.argmin(theta)

    return periods[best_idx], periods, theta

In [101]:
def analyze_star(t, y, dy, star_id):

    if len(t) < 10:
        return None

    mask = dy < np.percentile(dy, 90)
    t, y = t[mask], y[mask]

    if len(t) < 10:
        return None

    y = detrend(y)

    ls = LombScargle(t, y)
    f, p = ls.autopower(minimum_frequency=min_f,
                        maximum_frequency=max_f)

    best_idx = np.argmax(p)
    f0 = f[best_idx]

    period_ls = 1 / f0
    fap = ls.false_alarm_probability(np.max(p))

    peak_ratio = np.max(p) / np.median(p)

    sorted_p = np.sort(p)

    return {
        "id": star_id,
        "t": t,
        "y": y,
        "frequency": f,
        "power": p,
        "f0": f0,
        "period_ls": period_ls,
        "fap": fap,
        "peak_ratio": peak_ratio,
    }

In [102]:
min_f = 1/101
max_f = 1/0.1

results = []

for i in range(len(Ks_b278)):
    r = analyze_star(
        Ks_mjd_b278[i],
        Ks_b278[i],
        Ks_err_b278[i],
        Id_b278[i]
    )
    if r:
        results.append(r)

print("Total:", len(results))

Total: 304


In [103]:
filtered = [
    r for r in results
    if (r["peak_ratio"] > 8) and (r["fap"] < 1e-4)
]

best_stars = sorted(
    filtered,
    key=lambda r: (r["peak_ratio"]),
    reverse=True
)

selected_stars = best_stars[:50]

print("Seleccionadas:", len(selected_stars))

Seleccionadas: 50


In [104]:
def plot_ls(result, idx):

    t, y = result["t"], result["y"]
    f, p = result["frequency"], result["power"]

    fig, axs = plt.subplots(1,3, figsize=(15,4))

    axs[0].scatter(t, y, s=10)
    axs[0].invert_yaxis()

    axs[1].plot(f, p)

    phase = ((t - t[0]) / result["period_ls"]) % 1
    axs[2].scatter(phase, y, s=10)
    axs[2].invert_yaxis()

    plt.suptitle(f"Estrella {idx}")
    plt.savefig(f"{output_dir}/Estrella_{idx}.png")
    plt.close()

In [105]:
pdm_candidates = []

for i, r in enumerate(selected_stars, 1):

    p_pdm, periods, theta = find_period_pdm_refined(
        r["t"], r["y"], r["f0"])

    theta_min = np.min(theta)
    theta_ratio = np.median(theta) / theta_min

    r["period_pdm"] = p_pdm
    r["theta_ratio"] = theta_ratio
    r["periods_pdm"] = periods   # 🔥 NUEVO
    r["theta"] = theta           # 🔥 NUEVO
    r["ls_index"] = i  # 🔥 clave

    if theta_ratio > 1.5:
        pdm_candidates.append(r)

print("Buenas para PDM:", len(pdm_candidates))

Buenas para PDM: 10


In [106]:
pdm_best = sorted(
    pdm_candidates,
    key=lambda r: r["theta_ratio"],
    reverse=True
)

pdm_sample = pdm_best[:10]

In [107]:
def plot_pdm(result):

    t, y = result["t"], result["y"]
    f, p = result["frequency"], result["power"]

    period_ls = result["period_ls"]
    period_pdm = result["period_pdm"]

    # 🔥 necesitas esto guardado antes
    periods_pdm = result["periods_pdm"]
    theta = result["theta"]

    fig, axs = plt.subplots(1, 4, figsize=(20, 5))

    # =========================
    # (1) Lomb-Scargle (frecuencia)
    # =========================
    axs[0].plot(f, p)
    axs[0].set_title("Lomb-Scargle")
    axs[0].set_xlabel("Frecuencia (1/día)")
    axs[0].set_ylabel("Potencia")

    # =========================
    # (2) PDM (θ)
    # =========================
    mask = periods_pdm < 10

    if np.sum(mask) > 5:
        axs[1].plot(periods_pdm[mask], theta[mask])
    else:
        axs[1].plot(periods_pdm, theta)  # fallback
    axs[1].set_title("PDM (mínima entropía)")
    axs[1].set_xlabel("Periodo (días)")
    axs[1].set_ylabel("θ (dispersión)")

    # marcar mínimo
    #axs[1].axvline(period_pdm, linestyle='--')

    # =========================
    # (3) Curva LS
    # =========================
    phase_ls = ((t - t[0]) / period_ls) % 1
    axs[2].scatter(phase_ls, y, s=10)
    axs[2].invert_yaxis()
    axs[2].set_title("Fase (LS)")

    # =========================
    # (4) Curva PDM
    # =========================
    phase_pdm = ((t - t[0]) / period_pdm) % 1
    axs[3].scatter(phase_pdm, y, s=10)
    axs[3].invert_yaxis()
    axs[3].set_title("Fase (PDM)")

    plt.suptitle(
        f"Estrella {result['ls_index']} | "
        f"P_LS={period_ls:.4f} | P_PDM={period_pdm:.4f}"
    )

    ratio = period_pdm / period_ls
    print(f"Ratio PDM/LS = {ratio:.2f}")

    plt.tight_layout()
    plt.savefig(f"{output_dir}/PDM_Estrella_{result['ls_index']}.png", dpi=150)
    plt.close()

In [108]:
for i, r in enumerate(selected_stars, 1):
    plot_ls(r, i)

for r in pdm_sample:
    plot_pdm(r)

Ratio PDM/LS = 1.00
Ratio PDM/LS = 1.00
Ratio PDM/LS = 1.00
Ratio PDM/LS = 0.94
Ratio PDM/LS = 0.92
Ratio PDM/LS = 1.00
Ratio PDM/LS = 1.00
Ratio PDM/LS = 0.93
Ratio PDM/LS = 0.99
Ratio PDM/LS = 0.97
